# Solutions — React Developer Tools

Only look here after you've actually tried the exercises in `react_devtools.ipynb`.

### LESSON 34 — Exercise

**1. Between `App` and a `MemberCard`.** Three components, in this order:

```text
App → Experiment14 → Panel → MemberList → MemberCard
```

so `Experiment14`, `Panel` and `MemberList` sit between them. Note that `App` is there too —
the playground's own component, which every experiment is rendered inside.

**2. `Panel`'s props.** `title="Team"` and `children`. The second one is the interesting
answer: `children` appears in the props list like any other prop, because that is exactly what
it is (LESSON 5). The tool is showing you that "the stuff nested between the tags" was never
special — it was a prop the whole time.

**3. The only component with state.** `Experiment14`, and it has **two** pieces:

```text
query   ""      (empty, because nothing has been typed)
leadId  "m2"
```

Everything else in the tree is driven entirely by props. That is worth noticing: a tree of
five component types, and exactly one of them owns anything.

**4. Katherine's `isLead`.** `false`. It was decided by **`MemberList`**, which is the
component that writes `isLead={member.id === leadId}` when it maps over the members. The card
itself has no say — it only receives the answer.

**5. Searching `ada`.** Two `MemberCard` entries disappear, leaving one. The component count
drops from 8 to 6.

`MemberList` renders one `MemberCard` per item in the `members` prop, with `.map()` —
**LESSON 19**. Fewer items in the array means fewer children in the tree. `SearchBox`'s
`value` prop also changes from `""` to `"ada"`, and `Experiment14`'s `query` state with it.

**6. Searching `zzz`.** `MemberList` renders `<p>No one matches that search.</p>` instead of
the list — an **early return** when the array is empty (LESSON 18 and LESSON 21's empty
state). It is still a component, so it is still in the tree; it just produced different
elements this time. The tree makes the distinction visible in a way the DOM cannot: same
component, different output.

**7. A question the Elements panel cannot answer.** Any number of them, but the sharpest is:

> *Why* does this row have a star when the other two do not?

The Elements panel shows a `<span>` containing `★ Lead`. That is the *result*. It cannot tell
you that the span exists because `isLead` was `true`, that `isLead` was computed from
`leadId`, or that `leadId` lives in `Experiment14`. The DOM has no memory of which component
produced it or what data it was holding at the time.

Others that work: which component rendered this `<li>`? What is in this component's state?
What did its parent pass it?

**Common mistakes.**

- Reading the source file to answer the questions. Quicker today, useless on the code you did
  not write, which is the situation this tool exists for.
- Expecting the two trees to line up one-to-one. They do not, and the counts above prove it —
  8 components against 17 DOM elements.
- Treating `key` as one of the card's props. It is not data for your component — it is React's
  own bookkeeping (LESSON 20). Listing the props a mapped `MemberCard` actually receives gives
  `member` and `isLead`, and nothing else.
- Assuming a component with no state is uninteresting. Most of this tree has no state, and
  every one of those components is still doing a job you can read off its props.

### LESSON 34 — Mini challenge

**1. Which component renders the badge.** `MemberCard`, and the prop is `isLead`. Selecting
the three cards in turn shows `isLead: false`, `isLead: true`, `isLead: false` — and the
`true` is on the card whose `member` prop is Grace Hopper.

**2. Where it is calculated.** `MemberList`. It is not stored anywhere as `isLead`; it is
worked out per card while mapping:

```text
isLead  =  member.id === leadId
```

`MemberList`'s own props show both halves of that comparison: `members` (three objects, each
with an `id`) and `leadId`.

**3. The value that is actually wrong.** `leadId`, and it is owned by **`Experiment14`** as
state. Its value is `"m2"` — which is Grace Hopper's `id`. Ada Lovelace's `id` is `"m1"`.

So nothing is broken in `MemberCard` and nothing is broken in `MemberList`. Both did exactly
what they were told with the values they were given. The wrong value entered the tree at the
top, and everything below it faithfully passed on the consequences — which is the normal shape
of a React bug, and the reason "walk up until the value stops being wrong" is the technique
worth learning.

**4. Pressing "Make Ada the lead".** `leadId` becomes `"m1"`, and the change shows up in
**four** components:

```text
Experiment14   state leadId   "m2"  ->  "m1"
MemberList     prop  leadId   "m2"  ->  "m1"
MemberCard     prop  isLead   false ->  true      (Ada)
MemberCard     prop  isLead   true  ->  false     (Grace)
```

One state change, four visible consequences, and the tree shows all of them at once. Katherine
Johnson's card is unchanged — her `isLead` was `false` before and after.

**5. Why the Elements panel was no help.** Because the bug is not in the markup. The DOM is a
perfectly accurate rendering of the wrong data — every element is exactly where it should be,
the star is a real `<span>` in the right place, and nothing about it is malformed. There is
nothing for the Elements panel to reveal, because nothing there is wrong. The mistake is in a
value, and values are what the Components tab shows.

### LESSON 35 — Exercise

**Part 1 — the `<Profiler>` API.**

**1. A fresh load.** Both lines say `phase=mount`:

```text
   [profiler] id=counter phase=mount
   [profiler] id=sidebar phase=mount
```

`"mount"` is the *first* time a tree is put on screen — there was nothing there before, so
every element had to be created. `"update"` is every commit after that, where React already
has something to compare against. A reload starts the app from nothing, so nothing can be an
update yet.

**2. Pressing `+1` twice.** **Four** lines, two per press:

```text
   [profiler] id=counter phase=update
   [profiler] id=sidebar phase=update
   [profiler] id=counter phase=update
   [profiler] id=sidebar phase=update
```

Each press is one state change, which is one commit, which reports once per `<Profiler>` in
the tree. Both subtrees are inside the component whose state changed, so both are in every
commit.

**3. The third `<Profiler>`.** It **does** produce a line — one per press:

```text
   [profiler] id=counter  phase=update
   [profiler] id=sidebar  phase=update
   [profiler] id=controls phase=update
```

If you predicted "no", the likely reasoning was that the button does not change — and it does
not. But `<Profiler>` does not report *changes*, it reports *commits*, and the button lives
inside `Experiment15`, whose state changed. The whole component re-rendered, so everything it
returned was re-rendered with it, button included.

This is worth sitting with: the thing being measured is work React did, not difference React
found. A subtree that renders identically every time still appears in every commit.

**4. Five durations.** They will all be different, and some may be `0`. A run measured here
gave `0.3999999761581421` for one commit and a different value for the next.

You should not quote them because they are a measurement of *this machine, this browser, this
moment*, taken in a development build with profiling instrumentation attached — not a property
of the code. They are useful for comparing one thing against another in the same session
("this commit costs far more than that one"), and useless as absolute facts.

**Part 2 — the DevTools Profiler panel.** Answers depend on your extension version, so these
are the shapes to expect rather than exact wording:

1. **Which components appear.** `Experiment15`, `CountDisplay` and `ShoppingList` — the
   component whose state changed and everything it rendered. The panel groups this as one
   commit, matching the two `[profiler]` lines the API printed for the same press.
2. **Does it agree with the console?** It should. They are two readouts of the same commit:
   the API reports per `<Profiler>` you placed, the panel reports per component, so the panel
   is more detailed but tells the same story.
3. **Why did it render?** Recent versions can show a reason — for `ShoppingList` it will be
   that its parent rendered, not that its props changed. `items` is a module-level constant, so
   the same array is handed down every time; the render happened because `Experiment15`
   re-rendered, and that is the honest answer.

If your version does not offer the "why", note that and move on — it is a convenience, and the
reasoning above is available from the Components tab anyway.

**Common mistakes.**

- Confusing the two Profilers. `<Profiler>` is React; the panel is the extension. If you find
  yourself looking for `onRender` in the browser UI, or for a record button in your JSX, this
  is the confusion.
- Reading `actualDuration` as a benchmark. It is a development-build measurement with
  profiling overhead included. Production behaves differently, and profiling is *disabled in
  the production build by default*.
- Expecting a `<Profiler>` to report only when something inside it changed. It reports commits.
- Leaving `<Profiler>` in the code forever. It is an instrument — useful while you are asking a
  question, clutter once you have the answer. The panel needs no code at all, which is usually
  why you would reach for it instead.
- Treating `"mount"` as an error because it only appears once. It is the normal first commit.

### LESSON 35 — Mini challenge

**1. No — a re-render is not evidence of a problem.** It is React doing the thing it is
designed to do: something above the sidebar changed, so React called the sidebar's function to
find out what the UI should look like now. The answer may well be "exactly the same as
before", in which case React changes nothing on screen. A profile showing a component is a
statement that it *ran*, not that it was slow, wrong or wasteful.

**2. Two things you would need to know.**

- **Is it expensive?** A component rendering in a fraction of a millisecond is not worth
  anyone's afternoon, however often it happens. Compare it against the other commits in the
  same recording rather than against a number from somewhere else.
- **Is it avoidable, and does anyone notice?** There must be a real symptom — something felt
  slow, a frame was dropped, typing lagged. "It appears in the profile" is not a symptom.

Without both, any change is speculative, and the change itself has a cost: memoisation adds
code, adds comparisons of its own, and adds a way to be subtly wrong later.

**3. Rendered but the DOM did not change.** That tells you the *output* of the render was
identical, so React had nothing to apply — the commit cost nothing in DOM work, which is
usually the expensive half.

It does **not** tell you the render was free. Your component function still ran, along with
everything it calls: the `.map()`, any filtering or sorting, any work done at the top of the
component. A component can do a great deal of work and still produce identical output. The
profile is where you find out which of those two situations you are in.

**4. What to do first.** Do not memoise anything. Instead:

- Establish the symptom. What did a user actually experience, and when?
- Record that interaction in the **Profiler panel** and look at what the commit cost *relative
  to the rest of the recording*. If the sidebar is a rounding error next to something else,
  the sidebar was never the story.
- Use the **Components tab** (LESSON 34) to understand *why* the sidebar is in the commit at
  all — which value changed above it, and whether that value needed to change.

That last point is worth more than any optimisation, because the answer is often that a piece
of state is in the wrong place (LESSON 29) and the whole question dissolves.

The tools for deliberately preventing renders — `memo`, `useMemo`, `useCallback`, and the
React Compiler — are topic 23. They are deliberately taught *after* this lesson, because each
one is the wrong move applied to a problem nobody confirmed.